# Likelihood Ratio (LR) and Mixed Estimators for American Greeks

 
**Phase:** 3 - Advanced Greek Estimation Strategies

## Objective
The goal of this notebook is to implement and validate two advanced Monte Carlo Greek estimation techniques:
1. **Likelihood Ratio (LR) Method**: A method that differentiates the probability density function (Score Function) to handle discontinuities in payoffs or early-exercise boundaries.
2. **Mixed Estimator**: A "best-of-both-worlds" approach that combines the low-variance **Pathwise Delta** (from JH's work) with the theoretically robust **LR Gamma**.

We will compare these against the baselines established in previous notebooks to demonstrate variance reduction and accuracy.

## Method Summary

### 1. Likelihood Ratio (LR) Method
Unlike the Pathwise method which differentiates the payoff, the LR method differentiates the underlying transition density. For a Black-Scholes dynamics, the "Score Function" for the first step is used.
- **Delta Score**: $\frac{\partial \ln f}{\partial S_0} = \frac{Z_1}{S_0 \sigma \sqrt{\Delta t}}$
- **Gamma Score**: $\frac{\partial^2 \ln f}{\partial S_0^2} = \frac{Z_1^2 - 1 - Z_1 \sigma \sqrt{\Delta t}}{S_0^2 \sigma^2 \Delta t}$

Where $Z_1$ is the standard normal shock of the first time step.

### 2. The Mixed Estimator Strategy
While LR is unbiased for Gamma (where Pathwise fails due to the non-differentiable exercise boundary), it suffers from high variance in Delta. 
The **Mixed Estimator** uses:
- **Pathwise Estimator** for Delta/Vega (holding the exercise policy fixed, leveraging the Envelope Theorem).
- **LR Estimator** for Gamma.

In [1]:
%pwd

'/Users/fabian/Desktop/MAFN Semester 2/Numerical Methods in Finance/Project/notebooks'

In [2]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

src_path = Path("../src").resolve()

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from lsmc_greeks.pricer import LSMCConfig
from lsmc_greeks.greeks.likelihood import estimate_greeks_lr
from lsmc_greeks.greeks.mixed import estimate_greeks_mixed


# 3. Global Parameters
params = {
    "spot": 36.0,
    "strike": 40.0,
    "rate": 0.06,
    "sigma": 0.2,
    "maturity": 1.0
}

# High-precision config for benchmarking
config = LSMCConfig(n_paths=100000, n_steps_per_year=50, seed=42)

In [3]:
print("Running Likelihood Ratio Estimator...")
lr_res = estimate_greeks_lr(**params, config=config)

print("Running Mixed Estimator...")
mixed_res = estimate_greeks_mixed(**params, config=config)

# Prepare results for comparison
data = [
    {
        "Method": "Likelihood Ratio (LR)",
        "Delta": lr_res['delta_estimate'],
        "Delta SE": lr_res['delta_std_error'],
        "Gamma": lr_res['gamma_estimate'],
        "Gamma SE": lr_res['gamma_std_error']
    },
    {
        "Method": "Mixed (Pathwise + LR)",
        "Delta": mixed_res['delta_estimate'],
        "Delta SE": mixed_res['delta_std_error'],
        "Gamma": mixed_res['gamma_estimate'],
        "Gamma SE": mixed_res['gamma_std_error']
    }
]

df_results = pd.DataFrame(data)

Running Likelihood Ratio Estimator...
Running Mixed Estimator...


In [4]:
# Displaying the comparison table
display(df_results.style.highlight_min(subset=['Delta SE'], color='lightgreen'))

,Method,Delta,Delta SE,Gamma,Gamma SE
0,Likelihood Ratio (LR),-0.698177,0.016654,0.092405,0.023681
1,Mixed (Pathwise + LR),-0.688372,0.001267,0.092405,0.023681


## Conclusion

### Key Observations:
1. **Gamma Consistency**: The Gamma values for both LR and Mixed methods are identical (e.g., ~0.0924). This is expected because the Mixed estimator leverages the LR Score Function for second-order derivatives, and under the same random seed, the paths and stopping rules are perfectly matched.
2. **Variance Reduction in Delta**: The Mixed Estimator shows a dramatic reduction in Delta variance (Standard Error) compared to the pure LR method. By switching to a Pathwise approach for Delta, we eliminate the noise introduced by the $1/\sqrt{\Delta t}$ term in the LR score function.
3. **Efficiency**: The Mixed Estimator provides a complete Greek vector ($\Delta, \Gamma$) with high precision in a single simulation pass, making it superior to Finite Difference (which requires multiple 'bumps') and pure LR (which is too noisy for Delta).

### Final Recommendation:
For American options under LSMC, the **Mixed Estimator** should be the production default for calculating risk sensitivities.